In [3]:
import ast
import json
import pandas as pd

DATA_PATH = "movies_datasets/credits.csv"
pd.set_option("display.max_columns", 50)

movies = pd.read_csv("movies_datasets/movies_metadata.csv", low_memory=False)
credits = pd.read_csv("movies_datasets/credits.csv")
keywords = pd.read_csv("movies_datasets/keywords.csv")
links = pd.read_csv("movies_datasets/links_small.csv")
ratings = pd.read_csv("movies_datasets/ratings_small.csv")

for dataset, df in {
    "movies_metadata": movies,
    "credits": credits,
    "keywords": keywords,
    "links": links,
    "ratings": ratings,
}.items():
    print(f"{dataset:16s} {df.shape[0]:>8,} rows {df.shape[1]:>3} columns")

movies_metadata    45,466 rows  24 columns
credits            45,476 rows   3 columns
keywords           46,419 rows   2 columns
links               9,125 rows   3 columns
ratings           100,004 rows   4 columns


In [ ]:
def missing_values(df):
    result = pd.DataFrame(
        {
            "Missing": df.isna().sum(),
            "Percent": (df.isna().mean() * 100).round(2),
        }
    )
    result = result[result["Missing"] > 0].sort_values(by="Percent", ascending=False)
    return result


for dataset, df in {
    "movies_metadata": movies,
    "credits": credits,
    "keywords": keywords,
    "links": links,
    "ratings": ratings,
}.items():
    print(f"Missing values in the dataset {dataset}:")
    result = missing_values(df)
    if result.empty:
        print("No missing values")
    else:
        print(result)

Missing values in the dataset movies_metadata:
                       Missing  Percent
belongs_to_collection    40972    90.12
homepage                 37684    82.88
tagline                  25054    55.10
overview                   954     2.10
poster_path                386     0.85
runtime                    263     0.58
release_date                87     0.19
status                      87     0.19
imdb_id                     17     0.04
original_language           11     0.02
popularity                   5     0.01
revenue                      6     0.01
production_countries         3     0.01
production_companies         3     0.01
spoken_languages             6     0.01
title                        6     0.01
video                        6     0.01
vote_average                 6     0.01
vote_count                   6     0.01
Missing values in the dataset credits:
No missing values
Missing values in the dataset keywords:
No missing values
Missing values in the dataset links:
 

In [ ]:
datasets = [
    ("movies_metadata", movies, "id"),
    ("credits", credits, "id"),
    ("keywords", keywords, "id"),
    ("links", links, "movieId"),
    ("ratings", ratings, ["userId", "movieId"]),
]

for name, df, key in datasets:
    duplicate_ids = df[df.duplicated(key, keep=False)].sort_values(by=key)
    # Exact duplicated across all columns
    exact_duplicate_count = df.duplicated().sum()

    print(f"\n{name} dataset:")
    # Count duplicate ids and inspect how duplicated ids differ
    print(f"Exact duplicated rows: {exact_duplicate_count}")
    print(f"Rows belonging to duplicated {key}s: {len(duplicate_ids)}")
    print(f"Unique duplicated {key}s: {duplicate_ids.drop_duplicates(key).shape[0]}")

    if not duplicate_ids.empty:
        varying_columns = [
            col
            for col in duplicate_ids.columns
            if col not in (key if isinstance(key, list) else [key])
            and duplicate_ids.groupby(key)[col].nunique(dropna=False).gt(1).any()
        ]
        print(
            f"Columns that vary among duplicated ids: {varying_columns or 'no columns differ'}"
        )
        print("\nExample duplicated ids")
        print(duplicate_ids.groupby(key).size().sort_values(ascending=False).head(5))


movies_metadata dataset:
Exact duplicated rows: 17
Rows belonging to duplicated ids: 59
Unique duplicated ids: 29
Columns that vary among duplicated ids: ['popularity', 'vote_count']

Example duplicated ids
id
141971    3
10991     2
109962    2
110428    2
105045    2
dtype: int64

credits dataset:
Exact duplicated rows: 37
Rows belonging to duplicated ids: 87
Unique duplicated ids: 43
Columns that vary among duplicated ids: ['crew']

Example duplicated ids
id
141971    3
4912      2
3057      2
8767      2
9755      2
dtype: int64

keywords dataset:
Exact duplicated rows: 987
Rows belonging to duplicated ids: 1972
Unique duplicated ids: 985
Columns that vary among duplicated ids: no columns differ

Example duplicated ids
id
141971    3
159849    3
9375      2
381353    2
1998      2
dtype: int64

links dataset:
Exact duplicated rows: 0
Rows belonging to duplicated movieIds: 0
Unique duplicated movieIds: 0

ratings dataset:
Exact duplicated rows: 0
Rows belonging to duplicated ['user